In [0]:
from pyspark.sql import functions as F

# ---------------- CONFIG ----------------
BRONZE_JIRA = "workspace.slainte_bronze.jira_issues_bronze"
GOLD_DB     = "workspace.slainte_gold"
DIM_TTYPE   = f"{GOLD_DB}.dim_ticket_type"

# ---------------- SETUP ----------------
spark.sql(f"CREATE DATABASE IF NOT EXISTS {GOLD_DB}")

# ---------------- BUILD DIM ----------------
df_dim_ticket_type = (
    spark.table(BRONZE_JIRA)
    .select(
        F.col("user_id").alias("source_user_id"),
        F.col("project_key").alias("project"),
        F.trim(F.lower(F.col("ticket_type"))).alias("ticket_type_name")
    )
    .filter(
        F.col("user_id").isNotNull() &
        F.col("project_key").isNotNull() &
        F.col("ticket_type").isNotNull()
    )
    .distinct()
    .withColumn("ticket_type_id", F.monotonically_increasing_id() + 1)
    .select("ticket_type_id", "source_user_id", "project", "ticket_type_name")
)

# ---------------- WRITE ----------------
df_dim_ticket_type.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(DIM_TTYPE)

print("✅ dim_ticket_type created successfully")
